# Miniprot Window Spidroin Screen Across Species

This notebook only orchestrates the pipeline. All batch logic lives in `spider_silkome_module.miniprot_window_screen`, and each execution step is a single `run_cmd(...)` call.


## Imports


In [2]:
from datetime import datetime

from spider_silkome_module import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR
from spider_silkome_module import run_cmd


2026-05-27 12:34:25.143 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


## Configuration


In [3]:
task_basename = "miniprot_window_spidroin_screen"
task_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#task_name = f"{task_basename}_{task_timestamp}"
task_name = "miniprot_window_spidroin_screen_20260526_115407"

threads = 70
window_processes = 70
faidx_threads = 70
force = False
force_plots = False

# Set to repeated CLI flags for smoke tests, for example:
# species_filter_arg = "--species-filter 001.Allagelena_difficilis --species-filter 002.Anyphaena_bomiensis"
species_filter_arg = ""
max_species_arg = ""

flank_bp = 50_000
region_flank = 1_000
track_label_fraction = 0.35
per_mrna_track_height = 0.65
pygenometracks_font_size = 7
pygenometracks_width = 42

strong_positive = 0.60
strong_query_coverage = 0.80
rescue_positive = 0.55
rescue_query_coverage = 0.70
min_aa = 1_000

query_proteins = EXTERNAL_DATA_DIR / "spidroins_full_length_with_dataset1_rename.fasta"
raw_ref_dir = RAW_DATA_DIR / "01.ref_gff"
miniprot_evidence_dir = PROCESSED_DATA_DIR / "miniprot_output"
nhmmer_parsed_dir = PROCESSED_DATA_DIR / "nhmmer_search_parsed"
typing_results_dir = PROCESSED_DATA_DIR / "typing_results"

interim_root = INTERIM_DATA_DIR / task_name
processed_root = PROCESSED_DATA_DIR / task_name
species_manifest = interim_root / "species_manifest.tsv"
manifests_dir = interim_root / "manifests"

prepare_windows_summary = manifests_dir / "prepare_windows_summary.tsv"
run_miniprot_summary = manifests_dir / "run_miniprot_summary.tsv"
split_miniprot_summary = manifests_dir / "split_miniprot_summary.tsv"
parse_models_summary = manifests_dir / "parse_models_summary.tsv"
select_candidates_summary = manifests_dir / "select_candidates_summary.tsv"
export_selected_summary = manifests_dir / "export_selected_summary.tsv"
build_tracks_summary = manifests_dir / "build_tracks_summary.tsv"
pygenometracks_summary = manifests_dir / "pygenometracks_summary.tsv"

all_selected_tsv = processed_root / "all_species_selected_spidroin_mpid.tsv"
all_selected_faa = processed_root / "all_species_selected_spidroin_proteins.faa"
final_manifest = processed_root / "species_run_manifest.tsv"

force_arg = "--force" if force else ""
force_plots_arg = "--force" if force_plots else ""

print(f"task:      {task_name}")
print(f"interim:   {interim_root}")
print(f"processed: {processed_root}")


task:      miniprot_window_spidroin_screen_20260526_115407
interim:   /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407
processed: /home/gyk/project/spider_silkome/data/processed/miniprot_window_spidroin_screen_20260526_115407


## 1. Build Species Manifest


In [11]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.make_species_manifest "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} "
    f"--task-name {task_name} "
    f"--raw-ref-dir {raw_ref_dir} "
    f"--miniprot-evidence-dir {miniprot_evidence_dir} "
    f"--nhmmer-parsed-dir {nhmmer_parsed_dir} "
    f"--typing-results-dir {typing_results_dir} "
    f"{species_filter_arg} {max_species_arg} {force_arg}",
    [species_manifest],
    force=True,
)


2026-05-26 11:54:11.936 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


2026-05-26 11:54:11.995 | INFO     | __main__:main:113 - Creating species manifest: /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/species_manifest.tsv
2026-05-26 11:54:12.001 | DEBUG    | spider_silkome_module.miniprot_window_screen.common:find_genome_fasta:110 - Selected genome FASTA for 001.Allagelena_difficilis: /home/gyk/project/spider_silkome/data/raw/01.ref_gff/001.Allagelena_difficilis/LAg-10-m5.genome.fa.gz
2026-05-26 11:54:12.002 | DEBUG    | spider_silkome_module.miniprot_window_screen.common:find_genome_fasta:110 - Selected genome FASTA for 002.Anyphaena_bomiensis: /home/gyk/project/spider_silkome/data/raw/01.ref_gff/002.Anyphaena_bomiensis/LAn-01-Anbo-m1.genome.fa.gz
2026-05-26 11:54:12.003 | DEBUG    | spider_silkome_module.miniprot_window_screen.common:find_genome_fasta:110 - Selected genome FASTA for 003.Atypus_heterothecus: /home/gyk/project/spider_silkome/data/raw/01.ref_gff/003.Atypus_heterothecus/LAt-01-Athe-f2.genome.f

Manifest species: 100%|██████████| 134/134 [00:00<00:00, 957.23it/s]


## 2. Prepare Candidate Windows


In [12]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.prepare_windows_batch "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} "
    f"--flank-bp {flank_bp} "
    f"--window-processes {window_processes} "
    f"--faidx-threads {faidx_threads} {force_arg}",
    [prepare_windows_summary],
    force=True,
)


2026-05-26 11:54:15.872 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Build window BEDs:   0%|          | 0/134 [00:00<?, ?it/s]

2026-05-26 11:54:15.936 | INFO     | __main__:main:98 - Preparing windows for 134 species; window_processes=70; faidx_threads=70
2026-05-26 11:54:16.029 | WARNING  | spider_silkome_module.miniprot_window_screen.prepare_windows:parse_evidence_intervals:36 - Missing nhmmer GFF: /home/gyk/project/spider_silkome/data/processed/nhmmer_search_parsed/027.Neobisiidae_sp/027.Neobisiidae_sp.gff
2026-05-26 11:54:16.031 | WARNING  | spider_silkome_module.miniprot_window_screen.prepare_windows:parse_evidence_intervals:36 - Missing nhmmer GFF: /home/gyk/project/spider_silkome/data/processed/nhmmer_search_parsed/041.Pseudogagrella_splendens/041.Pseudogagrella_splendens.gff
2026-05-26 11:54:16.035 | WARNING  | spider_silkome_module.miniprot_window_screen.prepare_windows:parse_evidence_intervals:36 - Missing nhmmer GFF: /home/gyk/project/spider_silkome/data/processed/nhmmer_search_parsed/044.Schizomida_sp/044.Schizomida_sp.gff
2026-05-26 11:54:16.035 | WARNING  | spider_silkome_module.miniprot_window_s

Extract window FASTA:  99%|█████████▉| 133/134 [00:46<00:00,  2.83it/s]

2026-05-26 11:55:07.513 | SUCCESS  | __main__:main:183 - Window preparation summary: /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/manifests/prepare_windows_summary.tsv


Extract window FASTA: 100%|██████████| 134/134 [00:46<00:00,  2.87it/s]


## 3. Run Miniprot


In [13]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.run_miniprot_batch "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} "
    f"--query-proteins {query_proteins} "
    f"--threads {threads} {force_arg}",
    [run_miniprot_summary],
    force=True,
)


2026-05-26 14:31:43.897 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Run miniprot:   0%|          | 0/134 [00:00<?, ?it/s][M::mp_ntseq_read@0.011*1.15] read 2828868 bases in 21 contigs
[M::mp_idx_build@0.011*1.15] 22124 blocks
[M::mp_idx_build@0.024*8.02] collected syncmers


2026-05-26 14:31:43.954 | INFO     | __main__:main:35 - Running miniprot for 134 species


[M::mp_idx_build@0.159*2.06] 1136006 kmer-block pairs
[M::mp_idx_print_stat] 376617 distinct k-mers; mean occ of infrequent k-mers: 3.02; 0 frequent k-mers accounting for 0 occurrences
[M::worker_pipeline::9.178*30.09] mapped 508 sequences
[M::main] Version: 0.18-r281
[M::main] CMD: miniprot --gff -t 70 -G 200000 --outc 0.05 --outs 0.10 --outn 1000 /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/001.Allagelena_difficilis/candidate_windows.fa /home/gyk/project/spider_silkome/data/external/spidroins_full_length_with_dataset1_rename.fasta
[M::main] Real time: 9.184 sec; CPU: 276.152 sec; Peak RSS: 17.011 GB
Run miniprot:   1%|          | 1/134 [00:09<20:30,  9.25s/it][M::mp_ntseq_read@0.012*1.14] read 3022121 bases in 16 contigs
[M::mp_idx_build@0.012*1.14] 23624 blocks
[M::mp_idx_build@0.027*7.10] collected syncmers
[M::mp_idx_build@0.167*2.00] 1200677 kmer-block pairs
[M::mp_idx_print_stat] 400662 distinct k-mers; mean occ of infrequent k-me

2026-05-26 15:01:18.914 | SUCCESS  | __main__:main:94 - Miniprot summary: /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/manifests/run_miniprot_summary.tsv


[M::worker_pipeline::19.365*34.23] mapped 508 sequences
[M::main] Version: 0.18-r281
[M::main] CMD: miniprot --gff -t 70 -G 200000 --outc 0.05 --outs 0.10 --outn 1000 /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/134.Zoropsis_spinimana/candidate_windows.fa /home/gyk/project/spider_silkome/data/external/spidroins_full_length_with_dataset1_rename.fasta
[M::main] Real time: 19.370 sec; CPU: 662.914 sec; Peak RSS: 58.139 GB
Run miniprot: 100%|██████████| 134/134 [29:34<00:00, 13.25s/it]


## 4. Split Miniprot GFF/PAF


In [14]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.split_miniprot_batch "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} {force_arg}",
    [split_miniprot_summary],
    force=True,
)


2026-05-26 15:15:39.823 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Split miniprot:   0%|          | 0/134 [00:00<?, ?it/s]
Split miniprot_vs_windows.gff: 3941it [00:00, 220296.83it/s]

Split miniprot_vs_windows.gff: 13421it [00:00, 221277.84it/s]

Split miniprot_vs_windows.gff: 2351it [00:00, 288437.38it/s]

Split miniprot_vs_windows.gff: 0it [00:00, ?it/s]

2026-05-26 15:15:39.908 | INFO     | __main__:main:33 - Splitting miniprot output for 134 species


Split miniprot_vs_windows.gff: 8272it [00:00, 206326.72it/s]
Split miniprot:   3%|▎         | 4/134 [00:00<00:04, 30.98it/s]
Split miniprot_vs_windows.gff: 4052it [00:00, 212534.48it/s]

Split miniprot_vs_windows.gff: 4062it [00:00, 196456.11it/s]

Split miniprot_vs_windows.gff: 3426it [00:00, 217015.56it/s]

Split miniprot_vs_windows.gff: 2155it [00:00, 215819.23it/s]

Split miniprot_vs_windows.gff: 2309it [00:00, 274741.79it/s]

Split miniprot_vs_windows.gff: 385it [00:00, 287588.07it/s]

Split miniprot_vs_windows.gff: 14951it [00:00, 231906.12it/s]
Split miniprot:   8%|▊         | 11/134 [00:00<00:02, 41.86it/s]
Split miniprot_vs_windows.gff: 1487it [00:00, 198824.64it/s]

Split miniprot_vs_windows.gff: 11731it [00:00, 302349.07it/s]

Split miniprot_vs_windows.gff: 3548it [00:00, 251362.10it/s]

Split miniprot_vs_windows.gff: 1538it [00:00, 226020.10it/s]

Split miniprot_vs_windows.gff: 8123it [00:00, 257677.16it/s]
Split miniprot:  12%|█▏        | 16/134 [00:00<00:02, 44.68it/s]
Sp

2026-05-26 15:15:43.496 | SUCCESS  | __main__:main:83 - Split summary: /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/manifests/split_miniprot_summary.tsv


Split miniprot_vs_windows.gff: 16025it [00:00, 389811.99it/s]

Split miniprot_vs_windows.gff: 8674it [00:00, 351225.99it/s]

Split miniprot_vs_windows.gff: 3128it [00:00, 247561.76it/s]

Split miniprot_vs_windows.gff: 6569it [00:00, 226626.83it/s]
Split miniprot:  99%|█████████▊| 132/134 [00:03<00:00, 37.09it/s]
Split miniprot_vs_windows.gff: 6591it [00:00, 239146.84it/s]

Split miniprot_vs_windows.gff: 9523it [00:00, 245871.75it/s]
Split miniprot: 100%|██████████| 134/134 [00:03<00:00, 37.40it/s]


## 5. Parse Miniprot Models


In [15]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.parse_models_batch "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} "
    f"--query-proteins {query_proteins} {force_arg}",
    [parse_models_summary],
    force=True,
)


2026-05-26 15:15:51.005 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


2026-05-26 15:15:51.319 | INFO     | __main__:main:36 - Parsing miniprot models for 134 species


Parse models:  99%|█████████▉| 133/134 [03:52<00:01,  1.71s/it]

2026-05-26 15:19:47.042 | SUCCESS  | __main__:main:84 - Parse models summary: /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/manifests/parse_models_summary.tsv


Parse models: 100%|██████████| 134/134 [03:55<00:00,  1.76s/it]


## 6. Select Candidate MPIDs


In [16]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.select_candidates_batch "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} "
    f"--strong-positive {strong_positive} "
    f"--strong-query-coverage {strong_query_coverage} "
    f"--rescue-positive {rescue_positive} "
    f"--rescue-query-coverage {rescue_query_coverage} "
    f"--min-aa {min_aa} {force_arg}",
    [select_candidates_summary],
    force=True,
)


2026-05-26 15:28:04.074 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Select typing-guided candidates:   0%|          | 0/23 [00:00<?, ?it/s]

2026-05-26 15:28:04.140 | INFO     | __main__:main:47 - Selecting candidates for 134 species



Select typing-guided candidates: 100%|██████████| 4/4 [00:00<00:00, 270.09it/s]

Select typing-guided candidates: 100%|██████████| 2/2 [00:00<00:00, 3393.45it/s]

Select typing-guided candidates: 100%|██████████| 4/4 [00:00<00:00, 104.05it/s]

Select candidates:  19%|█▉        | 26/134 [00:51<02:51,  1.59s/it]
Select paired-domain candidates: 0it [00:00, ?it/s]

Select unpaired-window candidates: 100%|██████████| 15/15 [00:00<00:00, 35992.31it/s]

Select typing-guided candidates: 100%|██████████| 5/5 [00:00<00:00, 830.59it/s]

Select candidates:  30%|██▉       | 40/134 [01:18<02:07,  1.36s/it]
Select paired-domain candidates: 0it [00:00, ?it/s]

Select unpaired-window candidates: 100%|██████████| 9/9 [00:00<00:00, 13462.46it/s]

Select typing-guided candidates: 100%|██████████| 9/9 [00:00<00:00, 378.70it/s]

Select paired-domain candidates: 0it [00:00, ?it/s]

Select candidates:  33%|███▎      | 44/134 [01:18<00:46,  1.94it/s]
Select paired-domain candidates: 0it [00:00, ?it/s]

Selec

2026-05-26 15:31:29.927 | SUCCESS  | __main__:main:128 - Selection summary: /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/manifests/select_candidates_summary.tsv



Select candidates: 100%|██████████| 134/134 [03:25<00:00,  1.54s/it]


## 7. Export Selected GFF3 And Protein FASTA


In [17]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.export_selected_batch "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} {force_arg}",
    [export_selected_summary],
    force=True,
)


2026-05-26 15:34:25.644 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


2026-05-26 15:34:25.906 | INFO     | __main__:main:38 - Exporting selected models for 134 species


Export selected:   0%|          | 0/134 [00:00<?, ?it/s]
Export selected_spidroin_models.gff3: 3941it [00:00, 258196.69it/s]
Export selected:   1%|          | 1/134 [00:00<00:15,  8.71it/s]
Export selected_spidroin_models.gff3: 13421it [00:00, 256922.66it/s]
Export selected:   1%|▏         | 2/134 [00:00<00:43,  3.02it/s]
Export selected_spidroin_models.gff3: 2351it [00:00, 285853.68it/s]

Export selected_spidroin_models.gff3: 8272it [00:00, 252353.19it/s]
Export selected:   3%|▎         | 4/134 [00:00<00:28,  4.50it/s]
Export selected_spidroin_models.gff3: 4052it [00:00, 256312.60it/s]
Export selected:   4%|▎         | 5/134 [00:01<00:24,  5.34it/s]
Export selected_spidroin_models.gff3: 4062it [00:00, 246342.05it/s]
Export selected:   4%|▍         | 6/134 [00:01<00:21,  5.87it/s]
Export selected_spidroin_models.gff3: 3426it [00:00, 257359.82it/s]

Export selected_spidroin_models.gff3: 2155it [00:00, 249937.10it/s]
Export selected:   6%|▌         | 8/134 [00:01<00:15,  8.08it/s]
Export

2026-05-26 15:34:49.406 | SUCCESS  | __main__:main:92 - Export summary: /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/manifests/export_selected_summary.tsv



Export selected_spidroin_models.gff3: 9523it [00:00, 260694.82it/s]
Export selected: 100%|██████████| 134/134 [00:23<00:00,  5.70it/s]


## 8. Aggregate Processed Results


In [18]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.aggregate_results "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} {force_arg}",
    [all_selected_tsv, all_selected_faa, final_manifest],
    force=True,
)


2026-05-26 15:34:55.235 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Aggregate species: 100%|██████████| 134/134 [00:00<00:00, 1485.35it/s]


2026-05-26 15:34:55.446 | SUCCESS  | __main__:main:138 - Aggregated selected_rows=1423 proteins=1423 manifest=/home/gyk/project/spider_silkome/data/processed/miniprot_window_spidroin_screen_20260526_115407/species_run_manifest.tsv


## 9. Build pyGenomeTracks Inputs


In [6]:
run_cmd(
    f"pixi run python -m spider_silkome_module.miniprot_window_screen.build_tracks_batch "
    f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} "
    f"--region-flank {region_flank} --clip-region-to-window "
    f"--track-label-fraction {track_label_fraction} "
    f"--per-mrna-track-height {per_mrna_track_height} "
    f"--plot-font-size {pygenometracks_font_size} "
    f"--plot-width {pygenometracks_width} {force_arg}",
    [build_tracks_summary],
    force=True,
)


2026-05-26 18:09:33.983 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Build tracks:   0%|          | 0/134 [00:00<?, ?it/s]

2026-05-26 18:09:34.042 | INFO     | __main__:main:39 - Building tracks for 134 species
2026-05-26 18:09:34.046 | INFO     | spider_silkome_module.miniprot_window_screen.build_tracks:main:47 - Building pyGenomeTracks configuration in /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/001.Allagelena_difficilis



Build tracks:   1%|          | 1/134 [00:00<01:08,  1.95it/s]

2026-05-26 18:09:34.550 | SUCCESS  | spider_silkome_module.miniprot_window_screen.build_tracks:main:112 - tracks=/home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/001.Allagelena_difficilis/pygenometracks/pygenometracks_commands.tsv; window_models=690; selected_models=16; nhmmer_hits=110; evidence_models=3030
2026-05-26 18:09:34.558 | INFO     | spider_silkome_module.miniprot_window_screen.build_tracks:main:47 - Building pyGenomeTracks configuration in /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/002.Anyphaena_bomiensis



Build tracks:   1%|▏         | 2/134 [00:02<02:35,  1.18s/it]

...


Build tracks:  99%|█████████▊| 132/134 [01:33<00:01,  1.61it/s]

2026-05-26 18:11:07.112 | SUCCESS  | spider_silkome_module.miniprot_window_screen.build_tracks:main:112 - tracks=/home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/132.Tibellus_oblongus/pygenometracks/pygenometracks_commands.tsv; window_models=1227; selected_models=12; nhmmer_hits=123; evidence_models=4202
2026-05-26 18:11:07.122 | INFO     | spider_silkome_module.miniprot_window_screen.build_tracks:main:47 - Building pyGenomeTracks configuration in /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/133.Xysticus_acerbus



Build tracks:  99%|█████████▉| 133/134 [01:33<00:00,  1.48it/s]

2026-05-26 18:11:07.909 | SUCCESS  | spider_silkome_module.miniprot_window_screen.build_tracks:main:112 - tracks=/home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/133.Xysticus_acerbus/pygenometracks/pygenometracks_commands.tsv; window_models=1155; selected_models=9; nhmmer_hits=142; evidence_models=4417
2026-05-26 18:11:07.922 | INFO     | spider_silkome_module.miniprot_window_screen.build_tracks:main:47 - Building pyGenomeTracks configuration in /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/134.Zoropsis_spinimana
2026-05-26 18:11:09.001 | SUCCESS  | spider_silkome_module.miniprot_window_screen.build_tracks:main:112 - tracks=/home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/134.Zoropsis_spinimana/pygenometracks/pygenometracks_commands.tsv; window_models=1473; selected_models=16; nhmmer_hits=200; evidence_models=6533
2026-05-26 18:11:09.017 | SUCCESS  |


Build tracks: 100%|██████████| 134/134 [01:34<00:00,  1.41it/s]


## 10. Run pyGenomeTracks Plots


In [4]:
# run_cmd(
#     f"pixi run python -m spider_silkome_module.miniprot_window_screen.run_pygenometracks_batch "
#     f"--species-manifest {species_manifest} --interim-root {interim_root} --processed-root {processed_root} {force_plots_arg}",
#     [pygenometracks_summary],
#     force=True,
# )
cmd = f"pixi run python -m spider_silkome_module.miniprot_window_screen.run_pygenometracks_batch \
    --species-manifest {species_manifest} \
    --interim-root {interim_root} \
    --processed-root {processed_root} \
    {force_plots_arg}"

print(f"Please run the following command:\n{cmd}")

Please run the following command:
pixi run python -m spider_silkome_module.miniprot_window_screen.run_pygenometracks_batch     --species-manifest /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407/species_manifest.tsv     --interim-root /home/gyk/project/spider_silkome/data/interim/miniprot_window_spidroin_screen_20260526_115407     --processed-root /home/gyk/project/spider_silkome/data/processed/miniprot_window_spidroin_screen_20260526_115407     
